# cachepy — Feature Showcase

**cachepy** is a disk-backed memoization decorator for Python.  
It caches function results to disk so expensive computations run only once.

### Contents

| # | Feature | What it shows |
|---|---------|---------------|
| 1 | [Basic Caching](#1.-Basic-Caching) | First call vs cache hit |
| 2 | [Argument Normalization](#2.-Argument-Normalization) | Positional, named, default equivalence |
| 3 | [kwargs Order Independence](#3.-kwargs-Order-Independence) | Dict key ordering ignored |
| 4 | [File Dependency Tracking](#4.-File-Dependency-Tracking) | Content-based invalidation |
| 5 | [Body Change Detection](#5.-Body-Change-Detection) | Function redefinition invalidates |
| 6 | [Version Parameter](#6.-Version-Parameter) | Manual cache-busting |
| 7 | [Force & Skip Save](#7.-Force-&-Skip-Save) | Per-call control |
| 8 | [External Dependencies](#8.-External-Dependencies) | `depends_on_files`, `depends_on_vars` |
| 9 | [Environment Variables](#9.-Environment-Variables) | `env_vars` tracking |
| 10 | [Verbose Mode](#10.-Verbose-Mode) | Logging cache decisions |
| 11 | [Cache Stats & Pruning](#11.-Cache-Statistics-&-Pruning) | Disk usage, cleanup |
| 12 | [Dependency Graph](#12.-Dependency-Graph) | Nested cached function tracking |
| 13 | [Recursion](#13.-Recursive-Functions) | Fibonacci with memoization |
| 14 | [Error Handling](#14.-Error-Handling) | Clean graph on failure |
| 15 | [YAML Config](#15.-YAML-Configuration) | Project-level settings |
| 16 | [Speed Benchmark](#16.-Speed-Benchmark) | First execution vs cache hit |

In [ ]:
# --- Setup & Styling ---
import os, sys, time, shutil
from pathlib import Path
from IPython.display import HTML, display

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

body, .rendered_html, .text_cell_render {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
    font-size: 15px !important;
    line-height: 1.65 !important;
    color: #1e293b !important;
}
.rendered_html h1 {
    font-weight: 700 !important; font-size: 2em !important;
    border-bottom: 3px solid #2563eb !important; padding-bottom: 0.3em !important;
    color: #0f172a !important;
}
.rendered_html h2 {
    font-weight: 600 !important; font-size: 1.4em !important;
    color: #1e40af !important; margin-top: 2em !important;
    border-bottom: 1px solid #e2e8f0 !important; padding-bottom: 0.2em !important;
}
.rendered_html h3 { font-weight: 600 !important; font-size: 1.1em !important; color: #334155 !important; }
.rendered_html code {
    font-family: 'JetBrains Mono', 'Fira Code', monospace !important;
    background: #f1f5f9 !important; padding: 2px 6px !important;
    border-radius: 4px !important; font-size: 0.88em !important; color: #7c3aed !important;
}
.rendered_html pre { font-family: 'JetBrains Mono', monospace !important; font-size: 0.88em !important; }
.rendered_html blockquote {
    border-left: 4px solid #2563eb !important; background: #eff6ff !important;
    padding: 12px 16px !important; margin: 12px 0 !important;
    border-radius: 0 8px 8px 0 !important; color: #1e40af !important;
}
.rendered_html table { border-collapse: collapse !important; margin: 12px 0 !important; font-size: 14px !important; }
.rendered_html th { background: #f1f5f9 !important; font-weight: 600 !important; }
.rendered_html th, .rendered_html td { padding: 6px 14px !important; border: 1px solid #e2e8f0 !important; }
div.input_area { border: 1px solid #e2e8f0 !important; border-radius: 6px !important; }
.CodeMirror { font-family: 'JetBrains Mono', monospace !important; font-size: 13.5px !important; }
</style>
"""))

# --- Imports ---
sys.path.insert(0, str(Path.cwd().parent))

from cachepy import cache_file, cache_tree_nodes, cache_tree_reset
from cachepy.cache_file import (
    cache_prune, cache_stats, fast_file_hash,
    _file_state_cache, load_config,
)

# --- Matplotlib style ---
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

COLORS = {
    "cold": "#dc2626",   # red  — first execution
    "hot":  "#16a34a",   # green — cache hit
    "accent": "#2563eb", # blue  — speedup / general
    "muted": "#64748b",  # slate — secondary
    "purple": "#7c3aed",
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica Neue", "Arial", "sans-serif"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": 600,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "figure.facecolor": "white",
    "figure.dpi": 100,
})

# --- Helper ---
DEMO_CACHE = Path("demo_cache")
def fresh_cache():
    if DEMO_CACHE.exists():
        shutil.rmtree(DEMO_CACHE)
    cache_tree_reset()
    _file_state_cache.clear()
    return DEMO_CACHE

print("Ready.")

---
## 1. Basic Caching

Wrap any function with `@cache_file(cache_dir)`. The first call executes normally;  
subsequent calls with the same arguments return instantly from disk.

> **What to observe:** The first call takes ~1 s (simulated work). The second call returns in < 1 ms.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def slow_computation(n):
    """Simulate an expensive computation."""
    time.sleep(1)
    return sum(i**2 for i in range(n))

In [ ]:
t0 = time.time()
result1 = slow_computation(10_000)
print(f"First call:  {result1:,}  ({time.time()-t0:.2f}s)")

t0 = time.time()
result2 = slow_computation(10_000)
print(f"Second call: {result2:,}  ({time.time()-t0:.4f}s)")

t0 = time.time()
result3 = slow_computation(5_000)
print(f"New args:    {result3:,}  ({time.time()-t0:.2f}s)")

> **Takeaway:** Same args = cache hit. Different args = new computation.

---
## 2. Argument Normalization

cachepy normalizes how arguments are passed — positional, named, or with explicit defaults all resolve to the same cache key.

> **What to observe:** Only calls 1 and 5 trigger execution (printed with `->`).

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def add(a, b, c=0):
    print(f"  -> executing add({a}, {b}, {c})")
    return a + b + c

print("Call 1: add(1, 2)");            add(1, 2)
print("Call 2: add(a=1, b=2)");        add(a=1, b=2)
print("Call 3: add(b=2, a=1)");        add(b=2, a=1)
print("Call 4: add(1, 2, c=0)");       add(1, 2, c=0)
print("Call 5: add(1, 2, c=10)");      add(1, 2, c=10)
print(f"\nCache entries: {len(list(cache_dir.glob('*.pkl')))}")

> **Takeaway:** `f(1, 2)`, `f(a=1, b=2)`, `f(b=2, a=1)`, and `f(1, 2, c=0)` all hit the same cache entry.

---
## 3. kwargs Order Independence

For `**kwargs` functions, keyword argument order is ignored (sorted internally).

> **What to observe:** Only one execution despite reversed key order.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def config_hash(**kwargs):
    print(f"  -> executing with {kwargs}")
    return str(sorted(kwargs.items()))

config_hash(alpha=0.1, beta=0.9)
config_hash(beta=0.9, alpha=0.1)  # cache hit
print(f"Cache entries: {len(list(cache_dir.glob('*.pkl')))}")

---
## 4. File Dependency Tracking

When arguments point to files, cachepy hashes **file content** (not just the path).  
Touching a file without changing content = cache hit. Changing content = cache miss.

> **What to observe:** Calls 1 and 3 execute; call 2 is a cache hit.

In [ ]:
cache_dir = fresh_cache()
data_file = Path("demo_data.csv")
data_file.write_text("gene,expr\nTP53,10.5\nBRCA1,8.2\n")

@cache_file(cache_dir, file_args=["fpath"])
def parse_csv(fpath):
    print(f"  -> parsing {fpath}")
    lines = Path(fpath).read_text().strip().split("\n")
    header = lines[0].split(",")
    return [dict(zip(header, l.split(","))) for l in lines[1:]]

print("Call 1 (original):")
print(f"  {parse_csv(str(data_file))}")

print("Call 2 (same content):")
print(f"  {parse_csv(str(data_file))}")

data_file.write_text("gene,expr\nTP53,10.5\nBRCA1,8.2\nEGFR,15.3\n")
_file_state_cache.clear()

print("Call 3 (file changed):")
print(f"  {parse_csv(str(data_file))}")

data_file.unlink()

---
## 5. Body Change Detection

Redefining a function (e.g. fixing a bug) changes its hash, automatically invalidating the cache.

> **What to observe:** Two separate cache entries — one per function body.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def transform(x):
    return x * 2

print(f"v1: transform(5) = {transform(5)}")

@cache_file(cache_dir)
def transform(x):
    return x * 3  # body changed

print(f"v2: transform(5) = {transform(5)}")
print(f"Cache entries: {len(list(cache_dir.glob('*.pkl')))}")

---
## 6. Version Parameter

Manually bump `version` to invalidate without touching the function body — useful for retrained models, updated reference data, etc.

> **What to observe:** Same function body, but version bump triggers re-execution.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir, version="1.0")
def predict(x):
    print("  -> running model")
    return x * 42

predict(10); predict(10)  # second is a hit

@cache_file(cache_dir, version="2.0")
def predict(x):
    print("  -> running model")
    return x * 42

predict(10)  # miss — different version

---
## 7. Force & Skip Save

Control caching per-call without changing the decorator:
- `_force=True` — always re-execute (ignores cache)
- `_skip_save=True` — execute but don't write to disk

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def fetch(query):
    print(f"  -> fetching '{query}'")
    return {"query": query, "ts": time.time()}

r1 = fetch("TP53")
r2 = fetch("TP53")                    # cache hit
print(f"Same timestamp: {r1['ts'] == r2['ts']}")

r3 = fetch("TP53", _force=True)       # forced re-run
print(f"New  timestamp: {r3['ts'] != r1['ts']}")

n_before = len(list(cache_dir.glob('*.pkl')))
fetch("BRCA1", _skip_save=True)       # dry run
n_after = len(list(cache_dir.glob('*.pkl')))
print(f"Files before/after skip_save: {n_before}/{n_after}")

---
## 8. External Dependencies

Declare files or variables outside the function signature that should invalidate the cache.

> **What to observe:** Changing the config file or the schema variable triggers re-execution.

In [ ]:
# --- depends_on_files ---
cache_dir = fresh_cache()
config = Path("demo_config.yml")
config.write_text("threshold: 0.05\n")

@cache_file(cache_dir, depends_on_files=[str(config)])
def analyze(x):
    print("  -> analyzing")
    return x ** 2

analyze(5);  analyze(5)  # hit
config.write_text("threshold: 0.01\n")
_file_state_cache.clear()
analyze(5)  # miss — config changed
config.unlink()

# --- depends_on_vars ---
cache_dir2 = fresh_cache()

@cache_file(cache_dir2, depends_on_vars={"schema": "v3"})
def process(x):
    print("  -> processing")
    return x + 1

process(10);  process(10)  # hit

@cache_file(cache_dir2, depends_on_vars={"schema": "v4"})
def process(x):
    print("  -> processing")
    return x + 1

process(10)  # miss — schema changed

---
## 9. Environment Variables

Track environment variables — the cache invalidates when they change.

In [ ]:
cache_dir = fresh_cache()
os.environ["GENOME_BUILD"] = "hg38"

@cache_file(cache_dir, env_vars=["GENOME_BUILD"])
def align(reads):
    build = os.environ.get("GENOME_BUILD", "unknown")
    print(f"  -> aligning to {build}")
    return f"aligned_{reads}_to_{build}"

align("sample1");  align("sample1")      # hit
os.environ["GENOME_BUILD"] = "hg19"
align("sample1")                          # miss
os.environ.pop("GENOME_BUILD", None)

---
## 10. Verbose Mode

Set `verbose=True` to log cache decisions (first execution / hit / miss).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="[cachepy] %(message)s", force=True)

cache_dir = fresh_cache()

@cache_file(cache_dir, verbose=True)
def compute(x):
    return x * 2

compute(1)   # first execution
compute(1)   # cache hit
compute(2)   # miss (different arg)

logging.getLogger().setLevel(logging.WARNING)

---
## 11. Cache Statistics & Pruning

Inspect disk usage and clean up old entries.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def big_result(n):
    return list(range(n))

for n in [100, 1000, 10_000, 50_000]:
    big_result(n)

stats = cache_stats(cache_dir)
print(f"Entries: {stats['n_entries']}  |  Size: {stats['total_size_mb']:.2f} MB")

cache_prune(cache_dir, days_old=0)
print(f"After prune: {len(list(cache_dir.glob('*.pkl')))} files")

---
## 12. Dependency Graph

When cached functions call other cached functions, cachepy tracks the call graph.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def load_data(path):
    return [1, 2, 3, 4, 5]

@cache_file(cache_dir)
def normalize(data):
    mean = sum(data) / len(data)
    return [x - mean for x in data]

@cache_file(cache_dir)
def pipeline(path):
    return normalize(load_data(path))

print(f"Result: {pipeline('input.csv')}\n")

for nid, node in cache_tree_nodes().items():
    p = len(node.get('parents', []))
    c = len(node.get('children', []))
    print(f"  {node['fname']:12s}  parents={p}  children={c}")

---
## 13. Recursive Functions

Recursive cached functions automatically memoize sub-problems.

> **What to observe:** First run calls fib 11 times (n=0..10). Second run: 0 calls.

In [ ]:
cache_dir = fresh_cache()
call_count = 0

@cache_file(cache_dir)
def fib(n):
    global call_count; call_count += 1
    if n <= 1: return n
    return fib(n-1) + fib(n-2)

call_count = 0
print(f"fib(10) = {fib(10)},  calls: {call_count}")

call_count = 0
print(f"fib(10) = {fib(10)},  calls: {call_count}  (all cached)")

---
## 14. Error Handling

Failed functions don't leave stale cache entries or broken graph nodes.

In [ ]:
cache_dir = fresh_cache()

@cache_file(cache_dir)
def risky(x):
    if x < 0: raise ValueError("negative")
    return x ** 0.5

print(f"risky(4) = {risky(4)}")

try:
    risky(-1)
except ValueError as e:
    print(f"risky(-1) raised: {e}")

print(f"Graph nodes: {len(cache_tree_nodes())}  |  Cache files: {len(list(cache_dir.glob('*.pkl')))}")

> **Takeaway:** 1 graph node + 1 cache file (only the successful call).

---
## 15. YAML Configuration

Load project-level defaults from a YAML file.

In [ ]:
config_path = Path("demo_cachepy.yml")
config_path.write_text(
    "cache_dir: /tmp/my_project_cache\n"
    "backend: pickle\n"
    "verbose: true\n"
    "env_vars:\n  - HOME\n  - GENOME_BUILD\n"
)

cfg = load_config(config_path)
for k, v in cfg.items():
    print(f"  {k}: {v}")

# Existing values take priority
cfg2 = load_config(config_path, existing={"backend": "custom"})
print(f"\nWith existing override: backend={cfg2['backend']}")

config_path.unlink()

---
## 16. Speed Benchmark

How much faster is a cache hit? We measure across different computation costs.

> **What to observe:** Cache overhead is constant (~1-3 ms) regardless of original computation time.

In [ ]:
sleep_times = [0.1, 0.25, 0.5, 1.0, 2.0]
first_times, cached_times = [], []

for delay in sleep_times:
    cd = fresh_cache()

    @cache_file(cd)
    def work(x, _delay=delay):
        time.sleep(_delay)
        return x ** 2

    t0 = time.time(); work(42); first_times.append(time.time() - t0)
    t0 = time.time(); work(42); cached_times.append(time.time() - t0)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

x = np.arange(len(sleep_times))
w = 0.35

ax = axes[0]
ax.bar(x - w/2, [t*1000 for t in first_times], w, label="First execution", color=COLORS["cold"])
ax.bar(x + w/2, [t*1000 for t in cached_times], w, label="Cache hit", color=COLORS["hot"])
ax.set_xlabel("Computation time")
ax.set_ylabel("Wall time (ms)")
ax.set_title("First Execution vs Cache Hit")
ax.set_xticks(x)
ax.set_xticklabels([f"{s}s" for s in sleep_times])
ax.legend(frameon=True, fancybox=True)
ax.set_yscale("log")

speedups = [f / c for f, c in zip(first_times, cached_times)]
ax2 = axes[1]
ax2.bar(x, speedups, color=COLORS["accent"], width=0.5)
ax2.set_xlabel("Computation time")
ax2.set_ylabel("Speedup (x)")
ax2.set_title("Cache Speedup Factor")
ax2.set_xticks(x)
ax2.set_xticklabels([f"{s}s" for s in sleep_times])
for i, s in enumerate(speedups):
    ax2.text(i, s + max(speedups)*0.03, f"{s:.0f}x", ha="center", fontweight="bold", fontsize=11)

fig.tight_layout()
plt.show()

print(f"Cache hit overhead: {np.mean(cached_times)*1000:.1f} ms avg")

---
## Cleanup

In [ ]:
if DEMO_CACHE.exists():
    shutil.rmtree(DEMO_CACHE)
print("Done.")